# XGBoost

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import math
import time
from datetime import timedelta
from tabulate import tabulate
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import f1_score, make_scorer, roc_auc_score, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight 
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    'ambl_lesions_radiomic' : FILE_PATH / 'ambl_lesions_radiomic_medsam.csv',
    'duke_lesions_radiomic' : FILE_PATH / 'duke_lesions_radiomic_medsam.csv',
    'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv',
    'duke_lesions' : FILE_PATH / 'duke_lesions.csv',
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training


In [2]:
def training(file_path, csv_name):

    df = pd.read_csv(file_path)

    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]', 'HER2 [SII]']

    # Vado a tenere i target che esistono del csv
    available_targets = [c for c in original_target_list if c in df.columns]

    # Se non ho nulla, esco
    if len(available_targets) == 0:
        print(f"[ERRORE] {csv_name} - Nessun target disponibile nel CSV.")
        return None

    # Richiedo i group
    if 'Patient ID' not in df.columns:
        raise ValueError(f"{csv_name} - manca 'Patient ID' necessario per Group split")

    df_validi = df.copy()

    final_target_list = []
    # Binarizzazione dei marker
    if 'PR [SII]' in available_targets:
        df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
        final_target_list.append('PR_class')

    if 'ER [SII]' in available_targets:
        df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
        final_target_list.append('ER_class')

    if 'KI67 [%]' in available_targets:
        df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
        final_target_list.append('KI67_class')

    if 'HER2 [SII]' in available_targets:
        her2_series = df_validi['HER2 [SII]']

        # Se la colonna è già binaria (0/1) -> uso quella
        her2_vals = her2_series.dropna().unique()
        if len(her2_vals) > 0 and set(np.unique(her2_vals)).issubset({0, 1, 0.0, 1.0}):
            df_validi['HER2_class'] = her2_series.astype('float')
        else:
            # Scala 0/1/2/3 -> positivo solo 3 (clinicamente coerente)
            # NB: 2+ è equivoco: se vuoi puoi escluderlo (vedi commento sotto)
            df_validi['HER2_class'] = (her2_series >= 3).astype('float')

            # --- opzionale: escludo gli equivoci 2+ (solo se her2 è 0-3)
            # mask_eq = (her2_series == 2)
            # df_validi.loc[mask_eq, 'HER2_class'] = np.nan
        final_target_list.append('HER2_class')

     # Toglo un target con 1 sola classe 
    targets_da_rimuovere = []
    for col in final_target_list:
        s = df_validi[col].dropna()
        if s.nunique() < 2:
            print(f"[ATTENZIONE] {csv_name} - Target {col} ha una sola classe. Lo escludo.")
            targets_da_rimuovere.append(col)

    for col in targets_da_rimuovere:
        final_target_list.remove(col)

    if len(final_target_list) == 0:
        print(f"[ERRORE] {csv_name} - Nessun target valido (>=2 classi).")
        return None

    # Prendo le righe con tutti i target presenti
    df_validi = df_validi.dropna(subset=final_target_list).copy()

    # converto a int dopo aver tolto NaN
    for col in final_target_list:
        df_validi[col] = df_validi[col].astype(int)

    # Feature cleaning
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE',
                        'isTN', 'Breast'] + available_targets + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    target = df_validi[final_target_list]
    groups = df_validi['Patient ID']

    features = features.fillna(features.mean())
    features.columns = [re.sub(r'\[|\]|<', '', col) for col in features.columns]

    if 'HER2_class' in final_target_list:
        y_strat = target['HER2_class'].astype(str)
        n_pos = int(target['HER2_class'].sum())
        # Se positivi pochi, 5 fold è troppo aggressivo.
        n_splits = 3 if n_pos < 10 else 5
    else:
        y_strat = target.astype(int).astype(str).agg('_'.join, axis=1)
        n_splits = 5

    # raggruppo combinazioni troppo rare per supportare n_splits
    vc = y_strat.value_counts()
    rare = vc[vc < n_splits].index
    y_strat = y_strat.where(~y_strat.isin(rare), other="RARE")

    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # split precomputati (così lo splitter vede y_strat e groups)
    splits = list(sgkf.split(features, y_strat, groups=groups))

    # Base model with essential fixed parameters
    base_model = XGBClassifier(
        random_state=42,
        n_jobs=1,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0,
        reg_lambda=1,
        scale_pos_weight=1,
    )

    multi_output_model = MultiOutputClassifier(base_model)

    iperparametri = {
        'estimator__n_estimators': [50, 75, 100],
        'estimator__max_depth': [2, 3, 4],
        'estimator__learning_rate': [0.05, 0.1, 1]
    }

    # Multi-output scorer
    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(
                f1_score(y_true[:, i], y_pred[:, i],
                         average='macro', zero_division=0)
            )
        return np.mean(scores)

    scorer = make_scorer(multi_f1_scorer)

    tot = (len(iperparametri['estimator__n_estimators'])
           * len(iperparametri['estimator__max_depth'])
           * len(iperparametri['estimator__learning_rate']))

    print(f"\nInizio Grid Search (GRID MINIMAL: {tot} combinazioni) per: {csv_name}")

    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=splits,             
        scoring=scorer,
        n_jobs=-1,
        verbose=1,
        return_train_score=False,
        error_score='raise'
    )

    grid_search.fit(features, target)

    best_params = grid_search.best_params_
    best_score  = grid_search.best_score_

    clean_best_params = {k.replace('estimator__', ''): v for k, v in best_params.items()}

    final_model_params = {
        'random_state': 42,
        'n_jobs': 1,
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 3,
        'reg_alpha': 0,
        'reg_lambda': 1,
        'scale_pos_weight': 1,
        **clean_best_params
    }


    # Metriche per FOLD e per LABEL
    fold_reports = []

    for train_idx, test_idx in splits:
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        model_clone = MultiOutputClassifier(XGBClassifier(**final_model_params))
        model_clone.fit(X_train, y_train)

        y_pred = model_clone.predict(X_test)
        y_proba_list = model_clone.predict_proba(X_test)

        fold_metrics = {}
        for i, col in enumerate(final_target_list):
            y_true_i = y_test.iloc[:, i]
            y_pred_i = y_pred[:, i]

            f1 = f1_score(y_true_i, y_pred_i, zero_division=0)
            acc = accuracy_score(y_true_i, y_pred_i)

            # AUC
            unique_classes = np.unique(y_true_i)
            if len(unique_classes) < 2:
                auc_val = np.nan
            else:
                try:
                    if y_proba_list[i].shape[1] == 2:
                        auc_val = roc_auc_score(y_true_i, y_proba_list[i][:, 1])
                    else:
                        auc_val = 0.5
                except ValueError:
                    auc_val = np.nan

            fold_metrics[col] = {
                'f1': f1,
                'accuracy': acc,
                'auc': auc_val
            }

        fold_reports.append(fold_metrics)

    final_result = [{
        **clean_best_params,
        'mean_score': best_score,
        'std_score': grid_search.cv_results_['std_test_score'][grid_search.best_index_],
        'fold_reports': fold_reports,
        'targets_used': final_target_list
    }]

    return final_result

# Vado a stampare il risultato in un formato leggibile

In [3]:
def print_grid_search_results(results_per_dataset):

    print("\n" + "=" * 80)
    print(" " * 20 + "Migliori metrice per ogni fold")
    print("=" * 80)

    for dataset_name, metrics_list in results_per_dataset.items():
        if not metrics_list:
            continue

        best_result = metrics_list[0]     # modello migliore per il dataset
        fold_reports = best_result["fold_reports"]


        print(f"\n\nDataset: {dataset_name}")
        print("-" * 80)

        target_names =  best_result.get("targets_used", [])

        for target in target_names:

            # --- LISTE DI VALORI SUI FOLD ---
            f1_list  = np.array([fold[target]["f1"] for fold in fold_reports])
            acc_list = np.array([fold[target]["accuracy"] for fold in fold_reports])
            auc_list = np.array([fold[target]["auc"] for fold in fold_reports], dtype=float)

            # --- BEST VALUES ---
            best_f1  = np.max(f1_list)
            best_acc = np.max(acc_list)

            # Per AUC rimuovo eventuali NaN
            valid_auc = ~np.isnan(auc_list)
            best_auc  = np.max(auc_list[valid_auc]) if valid_auc.any() else np.nan

            # --- STD DEV ---
            f1_std  = np.std(f1_list)
            acc_std = np.std(acc_list)
            auc_std = np.std(auc_list[valid_auc]) if valid_auc.any() else np.nan

            # --- STAMPO RISULTATI ---
            print(f"\nTarget: {target}")
            print(f"  F1-score     = {best_f1:.3f} " +" ± "+f" {f1_std:.3f}")
            print(f"  Accuracy     = {best_acc:.3f} " +" ± "+f" {acc_std:.3f}")
            print(f"  AUC          = {best_auc:.3f} " +" ± "+f" {auc_std:.3f}" if not np.isnan(best_auc) else
                  f"  AUC          = NaN           " +" ± "+f" NaN")


# Lettura dei file

In [4]:
start_time = time.time()


# Eseguo il training per tutti i dataset
results_per_dataset = {}
for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Usa la nuova funzione per stampare i risultati
print_grid_search_results(results_per_dataset)




end_time = time.time()
# Calcolo il tempo impiegato
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")



Inizio Grid Search (GRID MINIMAL: 27 combinazioni) per: ambl_lesions_radiomic
Fitting 3 folds for each of 27 candidates, totalling 81 fits
[ERRORE] duke_lesions_radiomic - Nessun target disponibile nel CSV.

Inizio Grid Search (GRID MINIMAL: 27 combinazioni) per: ambl_lesions
Fitting 3 folds for each of 27 candidates, totalling 81 fits
[ERRORE] duke_lesions - Nessun target disponibile nel CSV.

Inizio Grid Search (GRID MINIMAL: 27 combinazioni) per: t2_medsam
Fitting 3 folds for each of 27 candidates, totalling 81 fits

Inizio Grid Search (GRID MINIMAL: 27 combinazioni) per: t2_preprocessed
Fitting 3 folds for each of 27 candidates, totalling 81 fits

Inizio Grid Search (GRID MINIMAL: 27 combinazioni) per: t2_original
Fitting 3 folds for each of 27 candidates, totalling 81 fits

Inizio Grid Search (GRID MINIMAL: 27 combinazioni) per: medsam_dynamic
Fitting 3 folds for each of 27 candidates, totalling 81 fits

Inizio Grid Search (GRID MINIMAL: 27 combinazioni) per: preprocessed_dynamic